# 42. 데이터셋 구조와 Annotation 이해

이 노트북은 semantic segmentation 데이터셋이 어떤 파일 구조와 label 형식을 갖는지 정리합니다.

이번 노트북의 목표는 다음과 같습니다.

- image와 mask 파일 쌍의 구조를 이해합니다.
- color mask와 class id mask의 차이를 구분합니다.
- train/validation split을 만들 때 주의할 점을 확인합니다.

## 42-1. 기본 폴더 구조

실전에서는 보통 다음과 같은 구조를 사용합니다.

```text
dataset/
  images/
    train/
    val/
  masks/
    train/
    val/
```

중요한 점은 image와 mask가 같은 sample id로 연결되어야 한다는 것입니다.

In [ ]:
from pathlib import Path

sample_paths = [
    Path("dataset/images/train/0001.jpg"),
    Path("dataset/masks/train/0001.png"),
    Path("dataset/images/val/0101.jpg"),
    Path("dataset/masks/val/0101.png"),
]

for path in sample_paths:
    print(path)

## 42-2. Color mask와 label mask

사람이 보기 좋은 mask는 RGB 색으로 저장될 수 있습니다. 하지만 학습에 필요한 mask는 각 픽셀이 class id를 갖는 2D 배열입니다.

```text
color mask: H, W, 3
label mask: H, W
```

모델 loss에는 label mask가 들어갑니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

label_mask = np.zeros((8, 8), dtype=np.int64)
label_mask[1:5, 1:5] = 1
label_mask[3:7, 4:7] = 2

palette = np.array([
    [120, 130, 140],
    [230, 70, 60],
    [60, 120, 230],
], dtype=np.uint8)

color_mask = palette[label_mask]

print("label mask shape:", label_mask.shape)
print("color mask shape:", color_mask.shape)
print("label ids:", np.unique(label_mask))

In [ ]:
cmap = ListedColormap(["#78828c", "#e6463c", "#3c78e6"])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(label_mask, cmap=cmap, vmin=0, vmax=2)
axes[0].set_title("label mask")
axes[1].imshow(color_mask)
axes[1].set_title("color mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 42-3. RGB mask를 class id로 변환하기

외부 annotation tool에서 RGB mask를 만든 경우, 학습 전에 색상별 class id로 변환해야 합니다.

In [ ]:
color_to_id = {
    (120, 130, 140): 0,
    (230, 70, 60): 1,
    (60, 120, 230): 2,
}

converted = np.zeros(color_mask.shape[:2], dtype=np.int64)
for color, class_id in color_to_id.items():
    converted[(color_mask == color).all(axis=-1)] = class_id

print("converted equals label_mask:", np.array_equal(converted, label_mask))

## 42-4. Split에서 확인할 점

train/validation split은 image와 mask 쌍을 기준으로 해야 합니다.

- image만 shuffle하고 mask를 따로 shuffle하면 쌍이 깨집니다.
- 같은 장면에서 나온 거의 동일한 이미지가 train과 val에 동시에 들어가면 평가가 과대평가될 수 있습니다.
- class가 적은 데이터셋은 validation에 특정 class가 빠질 수 있습니다.

## 정리

- segmentation 데이터셋은 image-mask pair가 기본입니다.
- 학습용 mask는 RGB 이미지가 아니라 class id 배열이어야 합니다.
- 다음 노트북 `43_Image_Mask_Dataset과_DataLoader_구현.ipynb`에서는 PyTorch Dataset 형태로 이 구조를 구현합니다.